# Isothermal-Isobaric ensemble (NPT)

In the last sheet we covered obtaining temperature dependent properties. With the Isothermal-Isobaric ensemble, we can also reach a target pressure. This does not only allow finite pressures but one also needs to add the cell parameters as additional degrees of freedom. This allows the evaluation of equilibrium volumes at specific temperatures. Depending on the system, it can be required for accurate simulations to first determine the average cell parameters and then run production simulations in NVE or NVT for this constant equilibrium simulation cell. We will only cover one of the most commonly implemented barostat, the method of Parinello and Rahman. For alternatives available in ASE, view this [page](https://ase-lib.org/ase/md.html#constant-npt-simulations-the-isothermal-isobaric-ensemble).

## Method of Parinello and Rahman

This is the most commonly implemented barostat as detailed in this [paper](https://doi.org/10.1063/1.328693). It introduces a matrix $\boldsymbol{\Sigma} = \mathbf{h}^{-1}(\sigma - \mathbf{I}P_{\text{ext}})(\mathbf{h}^{\text{T}})^{-1}$ which contains a stress $\sigma$, an external pressure $P_{\text{ext}}$ and the cell matrix $\mathbf h$. From that, they derived a new set of equations of motion which again contains a mass parameter $W_g$ that needs to be chosen such that the barostat contribution toward the kinetic energy does not eclipse the actual kinetic energy of the system. This is commonly chosen via the bulk modulus. Additionally, the barostat needs to be coupled with a thermostat to truly create an isothermal-isobaric ensemble. In ASE this is coupled with a Nosé-Hoover thermostat within the `MelchionnaNPT` class.

In [ ]:
# same initialization as the last time
from mace.calculators import MACECalculator
import ase.io
import ase.units
from pathlib import Path
import torch
import numpy as np

base_path = Path.cwd().parents[1]

model_file_name = (
    base_path
    / "data"
    / "models"
    / "full_dataset"
    / "M_2_ell_2_16_16_cut_5"
    / "Cu2S_stagetwo.model"
)
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"


# set up the calculator
calc = MACECalculator(model_file_name, device=device)

# set the structure file for Cu2S
struct_file_name = base_path / "data" / "Cu2S_FCC" / "Cu2S_conv.extxyz"

struct = ase.io.read(struct_file_name)

In [ ]:
from ase.md.npt import MelchionnaNPT
from ase.build import make_supercell
from ase.md.velocitydistribution import (
    MaxwellBoltzmannDistribution,
    Stationary,
)

# you can comment this first block to continue the current run
atoms = struct.copy()
atoms = make_supercell(atoms, np.diag([2, 2, 2]))
atoms.calc = calc
seed = np.random.randint(0, 10000)
print("seed:", seed)
rng = np.random.default_rng(seed=seed)
dt = 2.0
T = 300
n_steps = 500
pressure = 1.5  # in GPa
pfactor = 90  # in GPa
ptime = 10  # in fs
ttime = 10
traj_path = (
    base_path
    / "data"
    / "MD_trajectories"
    / f"Cu2S_FCC_222_NPT_{T:.0f}_{pressure:.0f}GPa.traj"
)

MaxwellBoltzmannDistribution(atoms, temperature_K=T, rng=rng)
Stationary(atoms)

# block end

integrator = MelchionnaNPT(
    atoms,
    dt * ase.units.fs,
    temperature_K=T,
    externalstress=pressure * ase.units.GPa,
    ttime=ttime * ase.units.fs,
    pfactor=pfactor * ase.units.GPa * (ptime * ase.units.fs) ** 2,
    trajectory=traj_path,
    loginterval=10,
    append_trajectory=False,
)


def print_md_info(current_atoms=atoms, current_integrator: MelchionnaNPT = integrator):
    n_atoms = len(current_atoms)
    e_pot_per_atom = current_atoms.get_potential_energy() / n_atoms
    e_kin_per_atom = current_atoms.get_kinetic_energy() / n_atoms
    kinetic_temperature = e_kin_per_atom / (1.5 * ase.units.kB)
    density = np.sum(current_atoms.get_masses()) / current_atoms.get_volume()
    stress_voigt = current_atoms.get_stress(voigt=True)
    P_virial = -np.mean(stress_voigt[:3])
    mv2 = (
        atoms.get_masses()[:, np.newaxis] * current_atoms.get_velocities() ** 2
    )
    P_kinetic = np.sum(mv2) / (3 * current_atoms.get_volume())
    pressure = (P_kinetic + P_virial) / ase.units.GPa
    print(
        f"Step #{current_integrator.nsteps + 1}: "
        f"T = {kinetic_temperature} K, "
        f"E = {e_pot_per_atom + e_kin_per_atom} eV / atom, "
        f"Volume = {current_atoms.get_volume()} A^3",
        f"Density = {density} amu / A^3",
        f"Pressure = {pressure} GPa",
        flush=True,
    )


integrator.attach(print_md_info, interval=1, current_integrator=integrator)

integrator.run(n_steps)

Note, that we had to do some math to get the pressure. It is built from a virial and a kinetic term:

$P = P_\mathrm{kin} + P_\mathrm{virial} = \frac{1}{3V} \left( \sum_{i=1}^{N} m_i |\mathbf{v}_i|^2 
    + \sum_{i=1}^{N} \sum_{j>i} \mathbf{r}_{ij} \cdot \mathbf{f}_{ij} \right)$

Here, $\mathbf{f}_{ij}$ is the force vector on atom $i$ due to displacement of atom $j$ and $\mathbf{r}_{ij}$ the distance vector between two atoms. The second term represents the sum of the diagonal elements of the stress tensor.


TASK 1:

Now use the output trajectory to plot a number of key figures as a function of time: 
- pressure
- temperature
- density
- volume
- length of the lattice vectors
  
Questions: 
- is the pressure correct?
- is the temperature correct?
- how much did the volume change?
- is the average volume converged?


TASK 2:
- Try changing the supercell size - what changed?
- Try the experiment again to make the model predict unphysical conditions, which pressure or temperature did you find where the model can't keep up?

In [ ]:
import matplotlib.pyplot as plt
from ase.io.trajectory import Trajectory

## Practical example: thermal expansion coefficient

The thermal expansion coefficient can be obtained in a straight-forward manner when running simulations at several temperatures. Run the above simulations in a temperature range and output the time-averaged volumes. Account for a certain amount of simulation time. Then use the script below to fit the thermal expansion coefficient.

In [ ]:
# Here, set up MD simulations or collect the results for several temperatures from the NPT ensemble

import numpy as np

temperatures = np.array([])
T_volumes = np.array([])
params = np.polyfit(temperatures, T_volumes, 1)
print(params)
print(
    f"Volumetric thermal expansion coefficient: {(params[0])/np.mean(T_volumes)} 1/K"
)

plt.scatter(temperatures, T_volumes, label="data")
plt.plot(temperatures, np.poly1d(params)(temperatures), label="linear fit")
plt.xlabel("Temperature / K")
plt.ylabel(r"Volume $\mathrm{\AA}^3$")
plt.legend()